In [10]:
!git clone -b ch https://github.com/IssacAnand/Plants-have-cancer.git
%cd Plants-have-cancer/backend
!ls

Cloning into 'Plants-have-cancer'...
remote: Enumerating objects: 75251, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 75251 (delta 28), reused 30 (delta 8), pack-reused 75171 (from 2)
Receiving objects: 100% (75251/75251), 3.78 GiB | 16.68 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (75420/75420), done.
/content/Plants-have-cancer/Plants-have-cancer/Plants-have-cancer/backend
checkpoints  data  dataset.py  vit_test_colab.ipynb  vit_test.ipynb


In [11]:
import os
import json
from pathlib import Path
from datetime import datetime

import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import timm

from dataset import PlantWildDataset
from dotenv import load_dotenv

In [12]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

2.10.0+cu128
True
NVIDIA A100-SXM4-80GB


In [13]:
class MobileViT(nn.Module):
    """
    Full MobileViT fine-tune — all layers trainable.
    Uses a lower LR for the backbone vs the head to avoid
    overwriting pretrained weights too aggressively.
    """

    def __init__(self, num_classes: int,
                 model_name: str, dropout: float = 0.2):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        self.embed_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, num_classes),
        )

        for param in self.backbone.parameters():
            param.requires_grad = True

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model           : {model_name}")
        print(f"Total params    : {total / 1e6:.2f}M")
        print(f"Trainable params: {trainable / 1e6:.2f}M  "
              f"({100 * trainable / total:.1f}% unfrozen)")

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(images))

    def get_encoder(self):
        return self.backbone

    def get_param_groups(self, backbone_lr: float, head_lr: float):
        """
        Separate LRs for backbone and head.
        Backbone gets a lower LR to preserve pretrained weights.
        """
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr},
            {"params": self.head.parameters(),     "lr": head_lr},
        ]

In [14]:
def get_transforms(img_size: int = 320, train: bool = True):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

In [15]:
def train(model, train_loader, test_loader,
          epochs, backbone_lr, head_lr, device, save_dir,
          class_weights=None, resume_from=None):

    os.makedirs(save_dir, exist_ok=True)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=0.1,
    )

    optimizer = torch.optim.AdamW(
        model.get_param_groups(backbone_lr, head_lr),
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )

    scaler   = GradScaler("cuda")

    # ── Resume from checkpoint ────────────────────────────────────
    start_epoch = 1
    best_acc    = 0.0

    if resume_from and os.path.exists(resume_from):
        checkpoint = torch.load(resume_from, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        start_epoch = checkpoint["epoch"] + 1
        best_acc    = checkpoint["best_acc"]
        print(f"Resumed from epoch {checkpoint['epoch']} | "
              f"Best acc so far: {best_acc:.2f}%")
    else:
        print("Starting training from scratch")

    # ── Training loop ─────────────────────────────────────────────
    for epoch in range(start_epoch, epochs + 1):

        # ── Train ──────────────────────────────────────────────────
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            with autocast("cuda"):
                loss = criterion(model(images), labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        scheduler.step(epoch)

        # ── Evaluate ───────────────────────────────────────────────
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                with autocast("cuda"):
                    preds = model(images).argmax(1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        acc = 100 * correct / total

        print(f"Epoch {epoch:>3}/{epochs}  "
              f"Loss: {train_loss / len(train_loader):.4f}  "
              f"Test Acc: {acc:.2f}%")

        # ── Save full checkpoint ───────────────────────────────────
        if acc > best_acc:
            best_acc = acc
            torch.save({
                "epoch":           epoch,
                "model_state":     model.state_dict(),
                "encoder_state":   model.get_encoder().state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_acc":        best_acc,
            }, os.path.join(save_dir, "checkpoint.pt"))
            print(f"  ✓ Best checkpoint saved ({best_acc:.2f}%)")

    print(f"\nFinetuning complete — best Test Acc: {best_acc:.2f}%")

    # ── Extract encoder weights for multimodal pipeline ───────────
    checkpoint = torch.load(
        os.path.join(save_dir, "checkpoint.pt"), map_location=device
    )
    torch.save(
        checkpoint["encoder_state"],
        os.path.join(save_dir, "best_image_encoder.pt")
    )
    print(f"Encoder weights saved → {save_dir}/best_image_encoder.pt")

    return model

In [16]:
MODEL_NAME = "mobilevitv2_200"
IMAGES_DIR  = "./data/images/plantwild"
IMG_SIZE    = 320
BATCH_SIZE  = 16
EPOCHS      = 50
BACKBONE_LR = 1e-5
HEAD_LR     = 1e-3
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR  = f"./checkpoints/{MODEL_NAME}_{timestamp}"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

Device: cuda


In [17]:
train_ds = PlantWildDataset(IMAGES_DIR,
                            transform=get_transforms(IMG_SIZE, train=True),
                            split="train")
test_ds  = PlantWildDataset(IMAGES_DIR,
                            transform=get_transforms(IMG_SIZE, train=False),
                            split="test", label_map=train_ds.label_map)

train_ds.save_label_map("./data/label_map.json")

class_counts  = train_ds.get_class_counts()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
print(f"Class weights computed for {len(class_weights)} classes")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)

model = MobileViT(
    num_classes=len(train_ds.classes),
    model_name=MODEL_NAME,
    dropout=0.2,
)

model = train(
    model, train_loader, test_loader,
    epochs=EPOCHS,
    backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR,
    device=DEVICE,
    save_dir=SAVE_DIR,
    class_weights=class_weights,
)

PlantWildDataset | split=train | 89 classes | 14832 images
PlantWildDataset | split=test | 89 classes | 3708 images
Label map saved → ./data/label_map.json
Class weights computed for 89 classes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

Model           : mobilevitv2_200
Total params    : 17.52M
Trainable params: 17.52M  (100.0% unfrozen)


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


...

Epoch  48/50  Loss: 1.2607  Test Acc: 70.44%


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


Epoch  49/50  Loss: 1.2526  Test Acc: 70.90%
  ✓ Best encoder saved (70.90%)


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


Epoch  50/50  Loss: 1.2469  Test Acc: 70.33%

Finetuning complete — best Test Acc: 70.90%
